In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

df_train = pd.read_csv('../data/processed/train_meta.csv', index_col='ecg_id', low_memory=False)
scaler = StandardScaler()

df_train['age_scaled'] = scaler.fit_transform(df_train[['age']])

print(f"Średnia wieku po skalowaniu: {df_train['age_scaled'].mean():.2f}")

Średnia wieku po skalowaniu: -0.00


In [4]:
import scipy.signal as signal
import wfdb

def extract_heart_rate(ecg_lead, fs=100.0):
    squared_signal = ecg_lead ** 2
    peaks, _ = signal.find_peaks(squared_signal, distance=int(fs * 0.6))     # Zakłada się min. 0.6s między uderzeniami

    if len(peaks) > 1:
        rr_intervals = np.diff(peaks) / fs  # czas w sekundach
        hr = 60.0 / np.mean(rr_intervals)
    else:
        hr = 60.0 # Wartość domyślna w razie błędu

    return hr

# Test na losowym pacjencie
sample_record = df_train['filename_lr'].iloc[0]
sig, _ = wfdb.rdsamp('../data/raw/' + sample_record)
lead_I = sig[:, 0]

print(f"Wyliczone tętno pacjenta: {extract_heart_rate(lead_I):.1f} BPM")

Wyliczone tętno pacjenta: 63.9 BPM


In [6]:
import pywt
import numpy as np

def generate_log_cwt(ecg_lead, fs=100.0):
    scales = np.arange(1, 40)               # Skale dla falki (odpowiadają różnym częstotliwościom)
    dt = 1.0 / fs

    # Generowanie skalogramu (Zespolona falka Morleta w PyWavelets)
    # Funkcja pywt.cwt zwraca macierz współczynników oraz wektor rzeczywistych częstotliwości
    cwt_matrix, freqs = pywt.cwt(ecg_lead, scales, 'cmor1.5-1.0', sampling_period=dt)

    # Obliczenie mocy sygnału (moduł z macierzy zespolonej)
    cwt_power = np.abs(cwt_matrix)

    # Przekształcenie matematyczne: kompresja logarytmiczna (log(x + epsilon))
    # Epsilon (1e-7) zapobiega błędowi log(0)
    cwt_log = np.log(cwt_power + 1e-7)

    return cwt_log